In [1]:
import pandas as pd

In [4]:
!unzip '/content/IMDB Dataset.csv.zip' -d '/content/'

Archive:  /content/IMDB Dataset.csv.zip
  inflating: /content/IMDB Dataset.csv  


In [7]:
df=pd.read_csv('/content/IMDB Dataset.csv')
df=df.iloc[:20000]
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
19995,"ok. for starters, taxi driver is amazing. this...",negative
19996,"It's sort of hard for me to say it, because I ...",negative
19997,I still liked it though. Warren Beatty is only...,positive
19998,We could still use Black Adder even today. Ima...,positive


In [8]:
# to lowercase
# remove punc
# remove html tags
# remove stopwords
#lemmatize

In [10]:
df['review']=df['review'].str.lower()
df

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive
...,...,...
19995,"ok. for starters, taxi driver is amazing. this...",negative
19996,"it's sort of hard for me to say it, because i ...",negative
19997,i still liked it though. warren beatty is only...,positive
19998,we could still use black adder even today. ima...,positive


In [11]:
import re

def remove_html(text):
  pattern =re.compile('<*.?>')
  return pattern.sub(' ', text)

In [12]:
df['review']=df['review'].apply(remove_html)

In [13]:
import string

def remove_punc(text):
  return text.translate(str.maketrans('','', string.punctuation))

In [14]:
df['review']=df['review'].apply(remove_punc)

In [15]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

def remove_stopwords(text):
    stop_words = set(stopwords.words('english'))

    words = text.split()

    filtered_words = [
        word for word in words
        if word.lower() not in stop_words
    ]

    return " ".join(filtered_words)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [16]:
df['review']=df['review'].apply(remove_stopwords)

In [17]:
# Now text vectorization

In [18]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 64.5 MB/s eta 0:00:00


In [24]:
import nltk
import gensim
from nltk import sent_tokenize
from gensim.utils import simple_preprocess
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [22]:
data=[]

for word in df['review']:
  rs=sent_tokenize(word)
  for s in rs:
    data.append(simple_preprocess(s))

In [23]:
len(data)

20000

In [27]:
model=gensim.models.Word2Vec(
    window=5,
)

In [28]:
model.build_vocab(data)

In [29]:
model.train(data, total_examples=model.corpus_count, epochs=model.epochs)

(10969046, 12128345)

In [30]:
import numpy as np
def doc_vector(doc):
  doc=[word for word in doc.split() if word in model.wv.index_to_key]
  return np.mean(model.wv[doc], axis=0)

In [31]:
x=[]

for doc in df['review'].values:
  x.append(doc_vector(doc))

In [33]:
len(x)

20000

In [36]:
y=df['sentiment']

In [38]:
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()

y=le.fit_transform(y)

In [40]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test=train_test_split(x,y,test_size=0.25, random_state=5)

In [41]:
from sklearn.ensemble import RandomForestClassifier

In [42]:
rf=RandomForestClassifier()

In [43]:
rf.fit(x_train, y_train)

RandomForestClassifier()

In [44]:
ypred=rf.predict(x_test)

In [45]:
from sklearn.metrics import accuracy_score

print(accuracy_score(ypred, y_test))

0.8082
